# Assignment 2.2 — Diffusion Model from Scratch

**MNIST, PyTorch only, no diffusers, no bonus/class conditioning.**

Implements:
1. Linear & cosine noise scheduler
2. Forward process `q_sample`
3. Minimal U-Net + ResNet blocks + sinusoidal time embedding
4. Noise-prediction training with MSE
5. DDPM sampling loop

The trained model is saved as `diffusion_mnist.pth` for Assignment 2.3.


In [ ]:
# Cell 1: Kaggle P100 setup — MUST run before importing torch
# Tesla P100 = sm_60. Kaggle's newer default PyTorch may not support it.
# Install only torch + its required CUDA dependencies. No torchvision needed.

%pip install -q "torch==2.7.1" --index-url https://download.pytorch.org/whl/cu126


In [ ]:
# Cell 2: Imports and GPU compatibility check
import os
import math
import urllib.request
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))
    print("Supported arch:", torch.cuda.get_arch_list())

    assert "sm_60" in torch.cuda.get_arch_list(), (
        "Current PyTorch does not support Tesla P100 (sm_60)."
    )


In [ ]:
# Cell 3: Hyperparameters
batch_size = 128
epochs = 5
learning_rate = 2e-4

T = 200
schedule_type = "cosine"

base_channels = 32
time_dim = 128


In [ ]:
# Cell 4: Load MNIST directly — no torchvision
mnist_path = "mnist.npz"

if not os.path.exists(mnist_path):
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz",
        mnist_path
    )

with np.load(mnist_path) as data:
    x_train = data["x_train"]
    y_train = data["y_train"]

x_train = torch.from_numpy(x_train).float().unsqueeze(1) / 127.5 - 1.0
y_train = torch.from_numpy(y_train).long()

train_dataset = TensorDataset(x_train, y_train)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

print("Train samples:", len(train_dataset))


In [ ]:
# Cell 5: Noise scheduler
class NoiseScheduler:
    def __init__(self, T=200, schedule="linear", device="cpu"):
        if schedule == "linear":
            betas = torch.linspace(1e-4, 0.02, T)

        elif schedule == "cosine":
            s = 0.008
            steps = torch.arange(T + 1, dtype=torch.float32)

            alpha_bar = torch.cos(
                ((steps / T + s) / (1 + s)) * math.pi / 2
            ) ** 2

            alpha_bar = alpha_bar / alpha_bar[0]

            betas = 1 - alpha_bar[1:] / alpha_bar[:-1]
            betas = torch.clamp(betas, 1e-4, 0.999)

        else:
            raise ValueError("schedule must be 'linear' or 'cosine'")

        self.beta = betas.to(device)
        self.alpha = 1.0 - self.beta
        self.alpha_bar = torch.cumprod(self.alpha, dim=0)

        alpha_bar_prev = torch.cat([
            torch.ones(1, device=device),
            self.alpha_bar[:-1]
        ])

        self.posterior_variance = (
            self.beta
            * (1.0 - alpha_bar_prev)
            / (1.0 - self.alpha_bar)
        )

linear_scheduler = NoiseScheduler(T, "linear", device)
cosine_scheduler = NoiseScheduler(T, "cosine", device)

scheduler = NoiseScheduler(T, schedule_type, device)

print("Using:", schedule_type)
print("beta:", scheduler.beta.shape)
print("alpha:", scheduler.alpha.shape)
print("alpha_bar:", scheduler.alpha_bar.shape)


In [ ]:
# Cell 6: Forward process q_sample
def extract(values, t, x_shape):
    out = values.gather(0, t)
    return out.view(t.shape[0], *((1,) * (len(x_shape) - 1)))


def q_sample(x0, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)

    sqrt_alpha_bar = torch.sqrt(
        extract(scheduler.alpha_bar, t, x0.shape)
    )

    sqrt_one_minus_alpha_bar = torch.sqrt(
        1.0 - extract(scheduler.alpha_bar, t, x0.shape)
    )

    return (
        sqrt_alpha_bar * x0
        + sqrt_one_minus_alpha_bar * noise
    )


In [ ]:
# Cell 7: Minimal U-Net
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        scale = math.log(10000) / (half - 1)

        freq = torch.exp(
            torch.arange(half, device=t.device) * -scale
        )

        angles = t.float().unsqueeze(1) * freq.unsqueeze(0)

        return torch.cat([
            torch.sin(angles),
            torch.cos(angles)
        ], dim=1)


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()

        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)

        self.norm1 = nn.GroupNorm(8, out_ch)
        self.norm2 = nn.GroupNorm(8, out_ch)

        self.time_proj = nn.Linear(time_dim, out_ch)

        self.residual = (
            nn.Conv2d(in_ch, out_ch, 1)
            if in_ch != out_ch
            else nn.Identity()
        )

    def forward(self, x, t_emb):
        h = F.silu(self.norm1(self.conv1(x)))
        h = h + self.time_proj(t_emb)[:, :, None, None]
        h = F.silu(self.norm2(self.conv2(h)))

        return h + self.residual(x)


class SimpleUNet(nn.Module):
    def __init__(self, base_channels=32, time_dim=128):
        super().__init__()

        self.time_embedding = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim)
        )

        self.input_conv = nn.Conv2d(1, base_channels, 3, padding=1)

        self.down1 = ResBlock(
            base_channels, base_channels, time_dim
        )

        self.downsample1 = nn.Conv2d(
            base_channels, base_channels * 2,
            4, stride=2, padding=1
        )

        self.down2 = ResBlock(
            base_channels * 2,
            base_channels * 2,
            time_dim
        )

        self.downsample2 = nn.Conv2d(
            base_channels * 2,
            base_channels * 4,
            4, stride=2, padding=1
        )

        self.middle = ResBlock(
            base_channels * 4,
            base_channels * 4,
            time_dim
        )

        self.upsample1 = nn.ConvTranspose2d(
            base_channels * 4,
            base_channels * 2,
            4, stride=2, padding=1
        )

        self.up1 = ResBlock(
            base_channels * 4,
            base_channels * 2,
            time_dim
        )

        self.upsample2 = nn.ConvTranspose2d(
            base_channels * 2,
            base_channels,
            4, stride=2, padding=1
        )

        self.up2 = ResBlock(
            base_channels * 2,
            base_channels,
            time_dim
        )

        self.output_conv = nn.Conv2d(base_channels, 1, 1)

    def forward(self, x, t):
        t_emb = self.time_embedding(t)

        x = self.input_conv(x)

        skip1 = self.down1(x, t_emb)

        x = self.downsample1(skip1)
        skip2 = self.down2(x, t_emb)

        x = self.downsample2(skip2)
        x = self.middle(x, t_emb)

        x = self.upsample1(x)
        x = torch.cat([x, skip2], dim=1)
        x = self.up1(x, t_emb)

        x = self.upsample2(x)
        x = torch.cat([x, skip1], dim=1)
        x = self.up2(x, t_emb)

        return self.output_conv(x)


In [ ]:
# Cell 8: Model and optimizer
model = SimpleUNet(
    base_channels=base_channels,
    time_dim=time_dim
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate
)

print("Parameters:", sum(p.numel() for p in model.parameters()))


In [ ]:
# Cell 9: Training loop
loss_history = []

for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0

    for x0, _ in train_loader:
        x0 = x0.to(device)

        t = torch.randint(
            0, T,
            (x0.size(0),),
            device=device
        )

        noise = torch.randn_like(x0)
        xt = q_sample(x0, t, noise)

        predicted_noise = model(xt, t)

        loss = F.mse_loss(
            predicted_noise,
            noise
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    loss_history.append(avg_loss)

    print(
        f"Epoch [{epoch:02d}/{epochs}] "
        f"MSE Loss: {avg_loss:.6f}"
    )


In [ ]:
# Cell 10: Training loss
plt.figure(figsize=(6, 4))
plt.plot(
    range(1, epochs + 1),
    loss_history,
    marker="o"
)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Diffusion Training Loss")
plt.grid(True)
plt.show()


In [ ]:
# Cell 11: One DDPM reverse step
@torch.no_grad()
def p_sample(model, x, t):
    batch_size = x.size(0)

    t_batch = torch.full(
        (batch_size,),
        t,
        device=device,
        dtype=torch.long
    )

    eps_pred = model(x, t_batch)

    beta_t = scheduler.beta[t]
    alpha_t = scheduler.alpha[t]
    alpha_bar_t = scheduler.alpha_bar[t]

    mean = (
        1.0 / torch.sqrt(alpha_t)
    ) * (
        x
        - beta_t
        / torch.sqrt(1.0 - alpha_bar_t)
        * eps_pred
    )

    if t == 0:
        return mean

    noise = torch.randn_like(x)
    variance = scheduler.posterior_variance[t]

    return mean + torch.sqrt(variance) * noise


In [ ]:
# Cell 12: DDPM sampling
@torch.no_grad()
def sample_ddpm(model, n_samples=16):
    model.eval()

    x = torch.randn(
        n_samples, 1, 28, 28,
        device=device
    )

    for t in reversed(range(T)):
        x = p_sample(model, x, t)

    return ((x.clamp(-1, 1) + 1) / 2).cpu()


generated = sample_ddpm(model, 16)

fig, axes = plt.subplots(4, 4, figsize=(6, 6))

for i, ax in enumerate(axes.flat):
    ax.imshow(generated[i].squeeze(), cmap="gray")
    ax.axis("off")

plt.suptitle("DDPM Samples")
plt.tight_layout()
plt.show()


In [ ]:
# Cell 13: Save checkpoint for Assignment 2.3
torch.save(
    model.state_dict(),
    "diffusion_mnist.pth"
)

print("Saved: diffusion_mnist.pth")
